# 10.7 — StyleGAN & CycleGAN

StyleGAN and CycleGAN solve two different control problems inside generative modeling. StyleGAN asks how a generator can expose continuous controls over coarse shape, mid-level structure, and fine texture; CycleGAN asks how an unpaired translation model can change domain while preserving content by making the translation come back home.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build StyleGAN and CycleGAN one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is made inspectable with NumPy arrays and small plots. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, small linear algebra, and toy image tensors.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the tiny random tensors.

### 1. Style modulation: scale and bias are continuous controls

A plain generator layer turns hidden activations into another activation map. StyleGAN adds a style-dependent affine control, often summarized as $y=s\cdot x+b$. The important word is **continuous**: style is not just a class label like "cat" or "car"; it is a vector whose coordinates can smoothly scale and shift features such as pose, width, brightness, or texture strength.

In [ ]:
x_w = np.array([-2., -1., 0., 1., 2.])  # one toy activation channel.
scale_w = 1.5  # style scale s.
bias_w = 0.4  # style bias b.
y_w = scale_w * x_w + bias_w  # styled activation y = s*x + b.
print("x:", x_w)
print("styled y:", np.round(y_w, 2))
print("middle check:", scale_w * 2 + bias_w)
assert round(float(scale_w * 2 + bias_w), 3) == 3.4

▶ What you'll see: every activation is stretched by 1.5 and shifted upward by 0.4; the hand-check value is 3.4.

In [ ]:
recovered_w = (y_w - bias_w) / scale_w  # invert the affine style operation.
print("recovered x:", np.round(recovered_w, 2))
assert np.allclose(recovered_w, x_w)
plt.figure(figsize=(4.4, 3))
plt.plot(x_w, label="original x", marker="o")
plt.plot(y_w, label="styled y", marker="o")
plt.title("1: style scale and bias")
plt.legend()
plt.show()

▶ What you'll see: two lines with the same ordering, but the styled line is taller and shifted upward.

*Why it's done this way:* the affine form $s\cdot x+b$ is the smallest useful control knob: multiplication changes feature strength and addition changes the baseline. Because the map is continuous and invertible when $s\neq0$, a small style change causes a small output change instead of a hard label switch.

### 2. AdaIN: match channel statistics instead of copying pixels

Adaptive Instance Normalization (AdaIN) is a concrete way to inject style. For each channel, normalize content to zero mean and unit variance, then impose the style's target mean and standard deviation:

$$\operatorname{AdaIN}(x,\mu_s,\sigma_s)=\sigma_s\frac{x-\mu_x}{\sigma_x}+\mu_s.$$

This preserves the within-channel spatial pattern of the content but changes the channel's statistics — a toy version of "same layout, different texture/appearance."

In [ ]:
content_w = np.array([[[1., 2., 3.], [2., 3., 4.]], [[10., 11., 12.], [8., 9., 10.]]])  # channels x height x width.
mu_c_w = content_w.mean(axis=(1, 2), keepdims=True)
sig_c_w = content_w.std(axis=(1, 2), keepdims=True)
print("content channel means:", np.round(mu_c_w.ravel(), 3))
print("content channel stds:", np.round(sig_c_w.ravel(), 3))
assert np.allclose(np.round(mu_c_w.ravel(), 3), [2.5, 10.0])

▶ What you'll see: channel 0 has mean 2.5, while channel 1 has mean 10.0.

In [ ]:
style_mean_w = np.array([0.0, 5.0]).reshape(2, 1, 1)  # target means from a style vector.
style_std_w = np.array([2.0, 0.5]).reshape(2, 1, 1)  # target standard deviations from a style vector.
normalized_w = (content_w - mu_c_w) / (sig_c_w + 1e-8)
styled_w = style_std_w * normalized_w + style_mean_w
print("styled means:", np.round(styled_w.mean(axis=(1, 2)), 3))
print("styled stds:", np.round(styled_w.std(axis=(1, 2)), 3))
assert np.allclose(np.round(styled_w.mean(axis=(1, 2)), 3), [0.0, 5.0])
assert np.allclose(np.round(styled_w.std(axis=(1, 2)), 3), [2.0, 0.5])

▶ What you'll see: the output channel statistics equal the requested style statistics.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6.4, 2.8))
ax[0].imshow(content_w[0], cmap="viridis"); ax[0].set_title("content ch0")
ax[1].imshow(styled_w[0], cmap="viridis"); ax[1].set_title("AdaIN ch0")
plt.suptitle("2: same pattern, new statistics")
plt.show()

▶ What you'll see: the spatial ramp remains visible, but its numeric scale and offset are changed.

*Why it's done this way:* normalization removes the content channel's original mean and contrast, leaving a standardized pattern. Multiplying by $\sigma_s$ and adding $\mu_s$ then writes style into exactly the first two moments, which are cheap, differentiable summaries of appearance.

### 3. Style mixing: early layers steer coarse structure, late layers steer texture

StyleGAN uses different style controls at different generator layers. A toy generator can be written as a sum of coarse, middle, and fine basis patterns. If style A controls early layers and style B controls late layers, the output keeps A's broad shape while borrowing B's details.

In [ ]:
grid_w = np.linspace(0, 1, 64)
coarse_w = np.sin(2 * np.pi * grid_w)  # low-frequency structure.
mid_w = 0.45 * np.sin(6 * np.pi * grid_w)  # medium wiggles.
fine_w = 0.15 * np.sin(26 * np.pi * grid_w)  # high-frequency texture.
style_a_w = np.array([1.1, 0.2, 0.1])
style_b_w = np.array([0.35, 1.0, 1.8])
print("style A layer scales:", style_a_w)
print("style B layer scales:", style_b_w)
assert round(float(style_b_w[2] / style_a_w[2]), 1) == 18.0

▶ What you'll see: style A emphasizes coarse shape; style B strongly emphasizes fine texture.

In [ ]:
basis_w = np.vstack([coarse_w, mid_w, fine_w])
sample_a_w = style_a_w @ basis_w
sample_b_w = style_b_w @ basis_w
mixed_w = np.array([style_a_w[0], style_a_w[1], style_b_w[2]]) @ basis_w
print("A std:", round(float(sample_a_w.std()), 3), "B std:", round(float(sample_b_w.std()), 3), "mixed std:", round(float(mixed_w.std()), 3))
plt.figure(figsize=(6, 3))
plt.plot(sample_a_w, label="style A")
plt.plot(sample_b_w, label="style B", alpha=.75)
plt.plot(mixed_w, label="A coarse + B fine", linewidth=2)
plt.title("3: layer-wise style mixing")
plt.legend()
plt.show()

▶ What you'll see: the mixed curve follows style A's broad wave but carries more high-frequency detail from style B.

*Why it's done this way:* early generator layers have large spatial influence, so their styles alter layout-like factors; late layers touch small details, so their styles alter texture-like factors. Mixing by layer is a structural bias toward disentanglement, not a proof that factors are perfectly independent.

### 4. Cycle consistency: an unpaired translation must make the return trip

CycleGAN has two maps: $G:X\rightarrow Y$ and $F:Y\rightarrow X$. If $G$ changes a point from domain X to domain Y, then $F(G(x))$ should recover the original $x$. The cycle loss

$$L_{cyc}=\lVert F(G(x))-x\rVert_1$$

prevents a translator from changing content freely just to fool a domain discriminator.

In [ ]:
x_w = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])  # four simple X-domain content points.
A_w = np.array([[1.2, 0.2], [-0.1, 0.9]])  # forward linear translation.
t_w = np.array([2.0, -1.0])  # domain shift from X to Y.
y_w = x_w @ A_w.T + t_w  # G(x).
print("G(x) first two rows:\n", np.round(y_w[:2], 3))
assert np.allclose(np.round(y_w[1], 3), [3.2, -1.1])

▶ What you'll see: X points are rotated/scaled and shifted into a new Y-domain location.

In [ ]:
Ainv_w = np.linalg.inv(A_w)
x_back_w = (y_w - t_w) @ Ainv_w.T  # F(G(x)) with the true inverse.
cycle_l1_w = np.mean(np.abs(x_back_w - x_w))
print("cycle L1 with inverse F:", round(float(cycle_l1_w), 6))
assert round(float(cycle_l1_w), 6) == 0.0

▶ What you'll see: the exact inverse returns every point, so cycle loss is zero.

In [ ]:
bad_back_w = (y_w - t_w) @ (0.85 * Ainv_w).T  # a content-shrinking return map.
bad_cycle_w = np.mean(np.abs(bad_back_w - x_w))
print("cycle L1 with bad F:", round(float(bad_cycle_w), 3))
assert round(float(bad_cycle_w), 3) == 0.075
plt.figure(figsize=(4.4, 3.4))
plt.scatter(x_w[:, 0], x_w[:, 1], label="x", s=70)
plt.scatter(bad_back_w[:, 0], bad_back_w[:, 1], label="F_bad(G(x))", marker="x", s=70)
plt.title("4: cycle loss exposes drift")
plt.legend()
plt.axis("equal")
plt.show()

▶ What you'll see: the bad return points shrink toward the origin instead of landing on the original square.

*Why it's done this way:* unpaired data tells us what Y should look like, but not which $y$ pairs with each $x$. The return-trip constraint supplies that missing alignment pressure: a map may change style/domain, but it pays an L1 penalty whenever it destroys recoverable content.

### 5. Adversarial matching plus cycle loss: realism is not enough

CycleGAN still uses adversarial pressure so translated samples look like the target domain. But adversarial matching alone can map many different inputs to plausible target samples and forget content. The practical objective balances two terms: "look like Y" and "come back to x."

In [ ]:
x_line_w = np.linspace(-1, 1, 80)
y_real_w = 2.0 + 0.5 * x_line_w  # target-domain line centered near 2.
y_good_w = 2.0 + 0.5 * x_line_w  # realistic and content-preserving.
y_collapse_w = np.full_like(x_line_w, y_real_w.mean())  # realistic mean, destroyed content variation.
adv_good_w = abs(y_good_w.mean() - y_real_w.mean())
adv_collapse_w = abs(y_collapse_w.mean() - y_real_w.mean())
print("adversarial mean gap good/collapse:", round(float(adv_good_w), 3), round(float(adv_collapse_w), 3))
assert round(float(adv_good_w), 3) == 0.0 and round(float(adv_collapse_w), 3) == 0.0

▶ What you'll see: a weak adversarial statistic cannot distinguish the faithful map from the collapsed map.

In [ ]:
x_good_back_w = 2 * (y_good_w - 2.0)  # exact inverse of y=2+0.5x.
x_collapse_back_w = 2 * (y_collapse_w - 2.0)  # collapsed outputs all return to one point.
cyc_good_w = np.mean(np.abs(x_good_back_w - x_line_w))
cyc_collapse_w = np.mean(np.abs(x_collapse_back_w - x_line_w))
print("cycle L1 good/collapse:", round(float(cyc_good_w), 3), round(float(cyc_collapse_w), 3))
assert round(float(cyc_good_w), 3) == 0.0
assert round(float(cyc_collapse_w), 3) == 0.506

▶ What you'll see: cycle loss is zero for the faithful translation and large for the collapsed one.

In [ ]:
plt.figure(figsize=(5.4, 3))
plt.plot(x_line_w, y_real_w, label="real Y trend")
plt.plot(x_line_w, y_good_w, "--", label="G good")
plt.plot(x_line_w, y_collapse_w, ":", label="G collapse")
plt.title("5: adversarial match needs cycle pressure")
plt.legend()
plt.show()

▶ What you'll see: the collapsed translator lands in the right domain average but erases input variation.

*Why it's done this way:* adversarial loss defines distributional realism, while cycle loss defines content preservation. Their combination is what lets unpaired translation change style without using paired examples.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for toy tensors, vectorized losses, and small numerical checks.
import matplotlib.pyplot as plt  # Import Matplotlib for line plots, scatter plots, heatmaps, and diagnostics.
np.random.seed(0)  # Fix the global seed so every stochastic example is repeatable.

def style_modulate(x, scale, bias):  # Apply the affine StyleGAN-style modulation y = s*x + b.
    return np.asarray(scale) * np.asarray(x) + np.asarray(bias)  # Broadcast scale and bias across the activation.

def adain(content, style_mean, style_std, eps=1e-8):  # Implement Adaptive Instance Normalization from scratch.
    content = np.asarray(content, dtype=float)  # Ensure predictable floating-point statistics.
    mu = content.mean(axis=(-2, -1), keepdims=True)  # Compute one mean per channel over spatial positions.
    sig = content.std(axis=(-2, -1), keepdims=True)  # Compute one standard deviation per channel over spatial positions.
    return np.asarray(style_std).reshape(-1, 1, 1) * (content - mu) / (sig + eps) + np.asarray(style_mean).reshape(-1, 1, 1)  # Normalize content and impose style statistics.

def cycle_l1(x, x_back):  # Compute the CycleGAN cycle-consistency L1 loss.
    return float(np.mean(np.abs(np.asarray(x_back) - np.asarray(x))))  # Average absolute return-trip error.

def show_signal(signal, title):  # Plot one 1-D toy generated signal.
    plt.figure(figsize=(5, 3))  # Create a compact figure.
    plt.plot(signal, color="teal")  # Draw the signal as a line.
    plt.title(title)  # Add an explanatory title.
    plt.xlabel("position")  # Label the horizontal position axis.
    plt.ylabel("activation")  # Label the generated activation value.
    plt.show()  # Display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Apply one style scale and bias

**Goal.** Compute $y=sx+b$ for one activation vector, because StyleGAN control starts with affine modulation. We build it in 2 steps.

In [ ]:
x_b1 = np.array([-1., 0., 1., 2.])  # Define one toy activation channel.
scale_b1 = 1.5  # Define the style scale.
bias_b1 = 0.4  # Define the style bias.
print("activation x:", x_b1)  # Inspect the input before styling.
print("scale and bias:", scale_b1, bias_b1)  # Inspect the style parameters.

▶ What you'll see: a short activation vector and two continuous style parameters.

In [ ]:
y_b1 = style_modulate(x_b1, scale_b1, bias_b1)  # Apply y = s*x + b.
print("styled activation:", np.round(y_b1, 2))  # Inspect the styled values.
assert round(float(y_b1[-1]), 3) == 3.4  # Verify 1.5*2 + 0.4 = 3.4.
plt.figure(figsize=(4, 3))  # Create a compact comparison plot.
plt.plot(x_b1, marker="o", label="x")  # Plot the original activations.
plt.plot(y_b1, marker="o", label="styled")  # Plot the styled activations.
plt.title("Basic 1: affine style modulation")  # Title the plot.
plt.legend()  # Show line labels.
plt.show()  # Display the plot.

▶ What you'll see: the styled values are stretched and shifted but keep the same ordering.

👀 Takeaway: StyleGAN-style control can be as simple as a continuous scale and bias applied to features.

### Basic 2 — Recover an activation after modulation

**Goal.** Invert the affine style operation, because seeing the inverse clarifies why scale and bias are interpretable knobs. We build it in 2 steps.

In [ ]:
x_b2 = np.array([0., 1., 2.])  # Define a small activation vector.
scale_b2 = 2.0  # Use a nonzero scale so the map is invertible.
bias_b2 = -1.0  # Use a visible shift.
y_b2 = style_modulate(x_b2, scale_b2, bias_b2)  # Apply the style transform.
print("styled y:", y_b2)  # Inspect the forward result.

▶ What you'll see: the forward transform maps [0, 1, 2] to [-1, 1, 3].

In [ ]:
x_recovered_b2 = (y_b2 - bias_b2) / scale_b2  # Undo y = s*x + b.
print("recovered x:", x_recovered_b2)  # Inspect the inverse result.
assert np.allclose(x_recovered_b2, x_b2)  # Verify the inverse recovers the original activations.
plt.figure(figsize=(4, 3))  # Create a compact bar plot.
plt.bar(["max error"], [np.max(np.abs(x_recovered_b2 - x_b2))], color="seagreen")  # Plot recovery error.
plt.title("Basic 2: affine inverse error")  # Title the diagnostic.
plt.show()  # Display the plot.

▶ What you'll see: the recovery error is exactly zero for this nonzero scale.

👀 Takeaway: affine style modulation is easy to reason about because scale and bias have direct algebraic effects.

### Basic 3 — Measure channel statistics

**Goal.** Compute per-channel means and standard deviations, because AdaIN controls style through channel statistics. We build it in 2 steps.

In [ ]:
content_b3 = np.array([[[1., 2.], [3., 4.]], [[4., 4.], [6., 6.]]])  # Define two 2x2 channels.
mean_b3 = content_b3.mean(axis=(1, 2))  # Compute one mean per channel.
std_b3 = content_b3.std(axis=(1, 2))  # Compute one standard deviation per channel.
print("means:", np.round(mean_b3, 3))  # Inspect channel averages.
print("stds:", np.round(std_b3, 3))  # Inspect channel contrasts.
assert np.allclose(np.round(mean_b3, 3), [2.5, 5.0])  # Verify concrete means.

▶ What you'll see: each channel has its own mean and contrast.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact statistics chart.
plt.bar(["ch0 mean", "ch1 mean"], mean_b3, color="teal")  # Plot the channel means.
plt.title("Basic 3: per-channel means")  # Title the chart.
plt.ylabel("mean activation")  # Label the numeric scale.
plt.show()  # Display the chart.

▶ What you'll see: channel 1 has a larger average activation than channel 0.

👀 Takeaway: AdaIN needs channel-wise statistics because style is applied feature channel by feature channel.

### Basic 4 — Normalize one channel

**Goal.** Turn a channel into zero mean and unit standard deviation, because AdaIN first removes the content statistics. We build it in 2 steps.

In [ ]:
channel_b4 = np.array([[1., 2.], [3., 4.]])  # Define one content channel.
mean_b4 = channel_b4.mean()  # Compute its mean.
std_b4 = channel_b4.std()  # Compute its standard deviation.
normalized_b4 = (channel_b4 - mean_b4) / std_b4  # Standardize the channel.
print("normalized mean:", round(float(normalized_b4.mean()), 6))  # Inspect the new mean.
print("normalized std:", round(float(normalized_b4.std()), 6))  # Inspect the new standard deviation.

▶ What you'll see: the normalized channel has mean 0 and standard deviation 1.

In [ ]:
assert round(float(normalized_b4.mean()), 6) == 0.0  # Verify zero mean.
assert round(float(normalized_b4.std()), 6) == 1.0  # Verify unit standard deviation.
plt.figure(figsize=(4, 3))  # Create a small heatmap.
plt.imshow(normalized_b4, cmap="coolwarm")  # Visualize standardized values.
plt.colorbar(label="z-score")  # Add a z-score scale.
plt.title("Basic 4: normalized content channel")  # Title the heatmap.
plt.show()  # Display the heatmap.

▶ What you'll see: low original values become negative z-scores and high values become positive z-scores.

👀 Takeaway: normalization preserves relative spatial pattern while removing original brightness and contrast.

### Basic 5 — Run AdaIN on two channels

**Goal.** Impose target style means and standard deviations, because AdaIN writes style statistics onto content features. We build it in 3 steps.

In [ ]:
content_b5 = np.array([[[1., 2.], [3., 4.]], [[4., 4.], [6., 6.]]])  # Define two content channels.
style_mean_b5 = np.array([10., -2.])  # Choose target means.
style_std_b5 = np.array([0.5, 3.0])  # Choose target standard deviations.
print("target means:", style_mean_b5)  # Inspect requested style means.
print("target stds:", style_std_b5)  # Inspect requested style contrasts.

▶ What you'll see: style asks channel 0 to become tight around 10 and channel 1 to become wide around -2.

In [ ]:
styled_b5 = adain(content_b5, style_mean_b5, style_std_b5)  # Apply AdaIN from scratch.
print("output means:", np.round(styled_b5.mean(axis=(1, 2)), 3))  # Inspect achieved means.
print("output stds:", np.round(styled_b5.std(axis=(1, 2)), 3))  # Inspect achieved standard deviations.
assert np.allclose(np.round(styled_b5.mean(axis=(1, 2)), 3), [10.0, -2.0])  # Verify means.
assert np.allclose(np.round(styled_b5.std(axis=(1, 2)), 3), [0.5, 3.0])  # Verify stds.

▶ What you'll see: AdaIN output statistics match the style targets.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))  # Create side-by-side heatmaps.
ax[0].imshow(content_b5[0], cmap="viridis"); ax[0].set_title("content")  # Show original channel 0.
ax[1].imshow(styled_b5[0], cmap="viridis"); ax[1].set_title("AdaIN")  # Show styled channel 0.
plt.suptitle("Basic 5: content pattern with style stats")  # Title the figure.
plt.show()  # Display the heatmaps.

▶ What you'll see: the ramp pattern remains, but its numeric range is restyled.

👀 Takeaway: AdaIN changes feature statistics without needing paired target images.

### Basic 6 — Build coarse and fine basis patterns

**Goal.** Separate low-frequency and high-frequency patterns, because StyleGAN layer controls affect different visual scales. We build it in 2 steps.

In [ ]:
t_b6 = np.linspace(0, 1, 80)  # Create a 1-D image coordinate.
coarse_b6 = np.sin(2 * np.pi * t_b6)  # Define a low-frequency coarse structure.
fine_b6 = 0.2 * np.sin(24 * np.pi * t_b6)  # Define a high-frequency texture ripple.
print("coarse std:", round(float(coarse_b6.std()), 3))  # Inspect large-scale variation.
print("fine std:", round(float(fine_b6.std()), 3))  # Inspect small-scale variation.

▶ What you'll see: the fine texture has lower amplitude than the coarse structure.

In [ ]:
signal_b6 = coarse_b6 + fine_b6  # Combine structure and texture.
plt.figure(figsize=(5, 3))  # Create a line plot.
plt.plot(coarse_b6, label="coarse")  # Plot coarse component.
plt.plot(signal_b6, label="coarse + fine", alpha=.8)  # Plot combined component.
plt.title("Basic 6: coarse shape plus fine texture")  # Title the plot.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the combined signal follows the big wave but wiggles rapidly around it.

👀 Takeaway: layer-wise style control is meaningful because different layers can own different spatial scales.

### Basic 7 — Mix styles across layers

**Goal.** Combine coarse controls from one style with fine controls from another, because StyleGAN can swap appearance factors layer by layer. We build it in 2 steps.

In [ ]:
basis_b7 = np.vstack([np.sin(2 * np.pi * t_b6), 0.2 * np.sin(24 * np.pi * t_b6)])  # Reuse coarse and fine bases from Basic 6.
style_a_b7 = np.array([1.0, 0.2])  # Style A emphasizes coarse shape.
style_b_b7 = np.array([0.4, 2.0])  # Style B emphasizes fine texture.
mixed_style_b7 = np.array([style_a_b7[0], style_b_b7[1]])  # Take coarse from A and fine from B.
print("mixed layer scales:", mixed_style_b7)  # Inspect the layer-wise style choice.
assert np.allclose(mixed_style_b7, [1.0, 2.0])  # Verify the intended mix.

▶ What you'll see: the mixed style uses A's first coordinate and B's second coordinate.

In [ ]:
sample_mixed_b7 = mixed_style_b7 @ basis_b7  # Generate a mixed signal.
sample_a_b7 = style_a_b7 @ basis_b7  # Generate the pure A signal for comparison.
plt.figure(figsize=(5, 3))  # Create a comparison plot.
plt.plot(sample_a_b7, label="style A")  # Plot pure style A.
plt.plot(sample_mixed_b7, label="A coarse + B fine")  # Plot the mixed result.
plt.title("Basic 7: style mixing")  # Title the figure.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the mixed signal keeps A's broad shape but has stronger fine wiggles.

👀 Takeaway: style mixing demonstrates how continuous controls can be localized to generator layers.

### Basic 8 — Translate points between two domains

**Goal.** Apply a simple forward map $G:X\rightarrow Y$, because CycleGAN translates samples from one domain to another. We build it in 2 steps.

In [ ]:
x_b8 = np.array([[0., 0.], [1., 0.], [0., 1.]])  # Define three X-domain points.
A_b8 = np.array([[1.0, 0.5], [0.0, 1.0]])  # Define a small shear transform.
t_b8 = np.array([2.0, -1.0])  # Define a domain shift.
y_b8 = x_b8 @ A_b8.T + t_b8  # Translate X points into Y.
print("translated points:\n", np.round(y_b8, 3))  # Inspect G(x).
assert np.allclose(y_b8[1], [3.0, -1.0])  # Verify one concrete translated point.

▶ What you'll see: every point moves into a shifted, sheared Y-domain coordinate system.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a scatter plot.
plt.scatter(x_b8[:, 0], x_b8[:, 1], label="X")  # Plot original X points.
plt.scatter(y_b8[:, 0], y_b8[:, 1], label="G(X)")  # Plot translated Y points.
plt.title("Basic 8: domain translation")  # Title the plot.
plt.legend()  # Show labels.
plt.axis("equal")  # Keep geometry honest.
plt.show()  # Display the scatter plot.

▶ What you'll see: the translated points keep relative structure while moving to a new region.

👀 Takeaway: a domain translator should change appearance or coordinate system while preserving content relationships.

### Basic 9 — Compute a cycle-consistency loss

**Goal.** Measure $\lVert F(G(x))-x\rVert_1$, because CycleGAN uses return-trip error to preserve content without pairs. We build it in 2 steps.

In [ ]:
x_b9 = np.array([[0., 0.], [1., 0.], [0., 1.]])  # Define original content points.
y_b9 = x_b9 + np.array([2.0, -1.0])  # Define G as a simple translation.
x_back_b9 = y_b9 - np.array([1.8, -1.0])  # Define F with a small x-direction mistake.
print("returned points:\n", np.round(x_back_b9, 3))  # Inspect F(G(x)).

▶ What you'll see: every returned point is shifted by 0.2 in the first coordinate.

In [ ]:
loss_b9 = cycle_l1(x_b9, x_back_b9)  # Compute average absolute return-trip error.
print("cycle L1:", round(loss_b9, 3))  # Inspect the cycle penalty.
assert round(loss_b9, 3) == 0.1  # Three points x two coordinates average a 0.2 one-coordinate error to 0.1.
plt.figure(figsize=(4, 3))  # Create a cycle-error plot.
plt.bar(["cycle L1"], [loss_b9], color="crimson")  # Plot the loss value.
plt.title("Basic 9: return-trip error")  # Title the diagnostic.
plt.show()  # Display the plot.

▶ What you'll see: the nonzero bar quantifies how much content drift the cycle detects.

👀 Takeaway: cycle consistency penalizes translators that cannot reconstruct the original input.

### Basic 10 — Compare L1 and L2 cycle penalties

**Goal.** Compare absolute and squared return-trip errors, because CycleGAN commonly uses L1 to preserve structure without overemphasizing outliers. We build it in 2 steps.

In [ ]:
errors_b10 = np.array([0.0, 0.1, -0.2, 1.0])  # Define several return-trip coordinate errors.
l1_b10 = np.mean(np.abs(errors_b10))  # Compute average absolute error.
l2_b10 = np.mean(errors_b10 ** 2)  # Compute average squared error.
print("L1:", round(float(l1_b10), 3), "L2:", round(float(l2_b10), 3))  # Inspect both penalties.
assert round(float(l1_b10), 3) == 0.325  # Verify the L1 number.
assert round(float(l2_b10), 3) == 0.263  # Verify the L2 number.

▶ What you'll see: the single large error contributes very differently under absolute versus squared loss.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a comparison chart.
plt.bar(["mean |e|", "mean e²"], [l1_b10, l2_b10], color=["teal", "orange"])  # Plot the two losses.
plt.title("Basic 10: cycle penalty choices")  # Title the plot.
plt.show()  # Display the chart.

▶ What you'll see: L1 and L2 summarize the same errors with different sensitivity to large deviations.

👀 Takeaway: L1 cycle loss measures average content drift directly and is less dominated by one large coordinate than L2 gradients would be.

## 🟡 Easy

### Easy 1 — Implement AdaIN for a toy feature map

**Goal.** Write the full AdaIN calculation in one inspectable example, because StyleGAN's style control often acts by changing feature statistics. We build it in 3 steps.

In [ ]:
content_e1 = np.array([[[1., 3., 5.], [2., 4., 6.]], [[2., 2., 4.], [4., 6., 6.]]])  # Define a 2-channel feature map.
style_mean_e1 = np.array([-1.0, 3.0])  # Define requested style means.
style_std_e1 = np.array([1.5, 0.25])  # Define requested style standard deviations.
print("content shape:", content_e1.shape)  # Inspect channels, height, and width.

▶ What you'll see: a tiny 2×2×3 feature map that is small enough to audit.

In [ ]:
styled_e1 = adain(content_e1, style_mean_e1, style_std_e1)  # Apply AdaIN.
out_mean_e1 = styled_e1.mean(axis=(1, 2))  # Measure output means.
out_std_e1 = styled_e1.std(axis=(1, 2))  # Measure output standard deviations.
print("output means:", np.round(out_mean_e1, 3))  # Inspect achieved means.
print("output stds:", np.round(out_std_e1, 3))  # Inspect achieved stds.
assert np.allclose(np.round(out_mean_e1, 3), [-1.0, 3.0])  # Verify style means.
assert np.allclose(np.round(out_std_e1, 3), [1.5, 0.25])  # Verify style stds.

▶ What you'll see: the output statistics exactly match the style request.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.7))  # Create side-by-side channel plots.
ax[0].imshow(content_e1[0], cmap="viridis"); ax[0].set_title("content ch0")  # Show the original first channel.
ax[1].imshow(styled_e1[0], cmap="viridis"); ax[1].set_title("styled ch0")  # Show the styled first channel.
plt.suptitle("Easy 1: AdaIN output")  # Title the comparison.
plt.show()  # Display the figure.

▶ What you'll see: the first channel keeps its spatial ordering while taking on the requested style range.

👀 Takeaway: AdaIN is just normalize content, then write in style mean and style variance.

### Easy 2 — Interpolate between two styles

**Goal.** Move smoothly between two style vectors, because StyleGAN controls are continuous rather than discrete labels. We build it in 3 steps.

In [ ]:
x_e2 = np.linspace(-1, 1, 60)  # Define a base activation pattern.
style_left_e2 = np.array([0.5, -0.5])  # Define scale and bias for one style.
style_right_e2 = np.array([2.0, 0.8])  # Define scale and bias for another style.
alphas_e2 = np.linspace(0, 1, 5)  # Define interpolation weights.
print("alphas:", alphas_e2)  # Inspect interpolation positions.

▶ What you'll see: five evenly spaced style interpolation weights from 0 to 1.

In [ ]:
signals_e2 = []  # Store one styled signal per interpolation.
for alpha_e2 in alphas_e2:  # Sweep from left style to right style.
    style_e2 = (1 - alpha_e2) * style_left_e2 + alpha_e2 * style_right_e2  # Linearly interpolate style parameters.
    signals_e2.append(style_modulate(x_e2, style_e2[0], style_e2[1]))  # Apply the interpolated style.
print("first/last endpoint means:", round(float(signals_e2[0].mean()), 3), round(float(signals_e2[-1].mean()), 3))  # Inspect endpoint shifts.
assert round(float(signals_e2[-1].mean()), 3) == 0.8  # Verify the last bias controls the mean of symmetric x.

▶ What you'll see: the mean activation shifts smoothly as bias increases.

In [ ]:
plt.figure(figsize=(5, 3))  # Create an interpolation plot.
for idx_e2, signal_e2 in enumerate(signals_e2):  # Plot each interpolated signal.
    plt.plot(signal_e2, label=f"α={alphas_e2[idx_e2]:.2f}")  # Label by interpolation weight.
plt.title("Easy 2: smooth style interpolation")  # Title the plot.
plt.legend(fontsize=8)  # Show compact labels.
plt.show()  # Display the plot.

▶ What you'll see: the curves morph gradually from low-contrast/downshifted to high-contrast/upshifted.

👀 Takeaway: continuous style vectors let users steer generated features smoothly.

### Easy 3 — Mix coarse and fine style controls

**Goal.** Generate signals from layer-wise style scales, because StyleGAN can combine coarse structure from one latent code with texture from another. We build it in 3 steps.

In [ ]:
grid_e3 = np.linspace(0, 1, 100)  # Define positions in a toy 1-D image.
bases_e3 = np.vstack([np.sin(2 * np.pi * grid_e3), 0.4 * np.sin(8 * np.pi * grid_e3), 0.15 * np.sin(32 * np.pi * grid_e3)])  # Define coarse, mid, and fine bases.
style_a_e3 = np.array([1.0, 0.2, 0.1])  # Style A: mostly coarse.
style_b_e3 = np.array([0.3, 1.0, 1.6])  # Style B: mid and fine.
print("basis shape:", bases_e3.shape)  # Inspect layer count by positions.

▶ What you'll see: three layer-like basis patterns over 100 positions.

In [ ]:
mixed_e3 = np.array([style_a_e3[0], style_b_e3[1], style_b_e3[2]]) @ bases_e3  # Mix early style from A and later styles from B.
pure_a_e3 = style_a_e3 @ bases_e3  # Generate pure A.
pure_b_e3 = style_b_e3 @ bases_e3  # Generate pure B.
print("mixed std:", round(float(mixed_e3.std()), 3))  # Inspect the variation of the mixed sample.
assert round(float((style_b_e3[2] / style_a_e3[2])), 1) == 16.0  # Verify B has much stronger fine style.

▶ What you'll see: the fine layer from B is sixteen times stronger than A's fine layer.

In [ ]:
plt.figure(figsize=(5.4, 3))  # Create a layer-mixing plot.
plt.plot(pure_a_e3, label="pure A")  # Plot pure style A.
plt.plot(pure_b_e3, label="pure B", alpha=.7)  # Plot pure style B.
plt.plot(mixed_e3, label="A coarse + B details", linewidth=2)  # Plot mixed style.
plt.title("Easy 3: layer-wise style mixing")  # Title the figure.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the mixed output keeps A's broad phase while borrowing B's mid/fine variation.

👀 Takeaway: style mixing works because different layers correspond to different spatial scales.

### Easy 4 — Compute CycleGAN return-trip loss

**Goal.** Build $G$, $F$, and $L_{cyc}$ on points, because cycle consistency is the central unpaired-translation constraint. We build it in 3 steps.

In [ ]:
x_e4 = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])  # Define source-domain points.
A_e4 = np.array([[1.1, 0.2], [0.0, 0.9]])  # Define a forward transform.
t_e4 = np.array([2.0, 1.0])  # Define a domain shift.
y_e4 = x_e4 @ A_e4.T + t_e4  # Apply G(x).
print("G(x) mean:", np.round(y_e4.mean(axis=0), 3))  # Inspect translated domain location.

▶ What you'll see: the translated points live around a shifted target-domain mean.

In [ ]:
Ainv_e4 = np.linalg.inv(A_e4)  # Compute the inverse linear map.
x_back_e4 = (y_e4 - t_e4) @ Ainv_e4.T  # Apply F(G(x)).
loss_e4 = cycle_l1(x_e4, x_back_e4)  # Compute cycle consistency.
print("cycle L1:", round(loss_e4, 6))  # Inspect return-trip error.
assert round(loss_e4, 6) == 0.0  # Verify perfect inverse gives zero cycle loss.

▶ What you'll see: the exact inverse has zero return-trip penalty.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a scatter comparison.
plt.scatter(x_e4[:, 0], x_e4[:, 1], label="x")  # Plot original points.
plt.scatter(x_back_e4[:, 0], x_back_e4[:, 1], marker="x", label="F(G(x))")  # Plot returned points.
plt.title("Easy 4: perfect cycle")  # Title the plot.
plt.legend()  # Show labels.
plt.axis("equal")  # Preserve geometry.
plt.show()  # Display the scatter.

▶ What you'll see: the returned points sit exactly on top of the original points.

👀 Takeaway: cycle loss is zero only when the forward translation remains recoverable.

### Easy 5 — Balance adversarial and cycle terms

**Goal.** Combine a toy domain-matching term with cycle loss, because CycleGAN needs realism and content preservation at the same time. We build it in 3 steps.

In [ ]:
adv_losses_e5 = np.array([0.05, 0.10, 0.20])  # Three toy translators' domain-realism gaps.
cycle_losses_e5 = np.array([0.60, 0.05, 0.15])  # Their content return-trip errors.
lambda_cyc_e5 = 10.0  # CycleGAN often gives cycle loss a strong weight.
print("adv losses:", adv_losses_e5)  # Inspect realism terms.
print("cycle losses:", cycle_losses_e5)  # Inspect content terms.

▶ What you'll see: model 0 looks realistic but has poor cycle consistency, while model 1 preserves content best.

In [ ]:
total_e5 = adv_losses_e5 + lambda_cyc_e5 * cycle_losses_e5  # Compute combined objective.
best_e5 = int(np.argmin(total_e5))  # Select the lowest-loss translator.
print("total losses:", np.round(total_e5, 3))  # Inspect combined scores.
print("best model index:", best_e5)  # Inspect which tradeoff wins.
assert best_e5 == 1  # Verify the low-cycle-loss model wins with λ=10.

▶ What you'll see: the cycle term dominates enough that content-preserving model 1 is selected.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a loss comparison plot.
plt.bar(["M0", "M1", "M2"], total_e5, color="purple")  # Plot total objective values.
plt.title("Easy 5: adversarial + λ cycle")  # Title the chart.
plt.ylabel("combined loss")  # Label the objective scale.
plt.show()  # Display the plot.

▶ What you'll see: the shortest bar balances realism and return-trip consistency.

👀 Takeaway: CycleGAN's objective is a tradeoff: target-domain realism alone is not enough.

## 🔴 Advanced

### Advanced 1 — Detect a collapsed CycleGAN mapping

**Goal.** Compare a content-preserving translator with a collapsed translator, because adversarial realism can miss content destruction. We build it in 4 steps.

In [ ]:
x_a1 = np.linspace(-1, 1, 101)  # Define ordered source-domain content.
y_real_a1 = 2.0 + 0.5 * x_a1  # Define the target-domain trend.
y_good_a1 = 2.0 + 0.5 * x_a1  # Define a faithful translator.
y_bad_a1 = np.full_like(x_a1, np.mean(y_real_a1))  # Define a collapsed translator.
print("real mean:", round(float(y_real_a1.mean()), 3), "bad mean:", round(float(y_bad_a1.mean()), 3))  # Inspect mean matching.

▶ What you'll see: the collapsed translator matches the target mean exactly.

In [ ]:
adv_good_a1 = abs(y_good_a1.mean() - y_real_a1.mean())  # Mean-based realism gap for good translator.
adv_bad_a1 = abs(y_bad_a1.mean() - y_real_a1.mean())  # Mean-based realism gap for collapsed translator.
print("adv gaps:", round(float(adv_good_a1), 3), round(float(adv_bad_a1), 3))  # Inspect weak adversarial statistic.
assert round(float(adv_bad_a1), 3) == 0.0  # Verify collapse can pass this weak realism check.

▶ What you'll see: both translators score equally under the weak mean-matching adversarial proxy.

In [ ]:
x_back_good_a1 = 2 * (y_good_a1 - 2.0)  # Return through the inverse translator.
x_back_bad_a1 = 2 * (y_bad_a1 - 2.0)  # Return collapsed outputs.
cyc_good_a1 = cycle_l1(x_a1, x_back_good_a1)  # Measure good cycle loss.
cyc_bad_a1 = cycle_l1(x_a1, x_back_bad_a1)  # Measure bad cycle loss.
print("cycle losses:", round(cyc_good_a1, 3), round(cyc_bad_a1, 3))  # Inspect content preservation.
assert round(cyc_bad_a1, 3) == 0.505  # Verify collapsed content is penalized.

▶ What you'll see: cycle loss separates the faithful translator from the collapsed translator.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a translator comparison plot.
plt.plot(x_a1, y_good_a1, label="good G")  # Plot faithful translator.
plt.plot(x_a1, y_bad_a1, label="collapsed G")  # Plot collapsed translator.
plt.title("Advanced 1: collapse passes weak realism")  # Title the plot.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the collapsed line is plausible in average location but carries no input variation.

👀 Takeaway: cycle consistency is a content-preservation guardrail that adversarial distribution matching cannot provide by itself.

### Advanced 2 — Sweep the cycle-loss weight

**Goal.** Show how $\lambda_{cyc}$ changes model selection, because CycleGAN must tune the realism-versus-content tradeoff. We build it in 4 steps.

In [ ]:
adv_a2 = np.array([0.02, 0.09, 0.18])  # Realism gaps for collapse, balanced, and identity-like models.
cyc_a2 = np.array([0.50, 0.08, 0.01])  # Cycle errors for the same models.
lambdas_a2 = np.array([0.0, 0.5, 1.0, 5.0, 10.0])  # Sweep cycle weights.
print("lambda grid:", lambdas_a2)  # Inspect tradeoff weights.

▶ What you'll see: λ ranges from ignoring cycle loss to strongly enforcing it.

In [ ]:
choices_a2 = []  # Store the best model index for each λ.
for lam_a2 in lambdas_a2:  # Loop over cycle weights.
    total_a2 = adv_a2 + lam_a2 * cyc_a2  # Compute combined objective.
    choices_a2.append(int(np.argmin(total_a2)))  # Store the best model.
print("best model by lambda:", choices_a2)  # Inspect the selected tradeoff.
assert choices_a2[0] == 0 and choices_a2[-1] == 2  # Verify low λ favors realism and high λ favors cycle.

▶ What you'll see: the selected model changes as cycle consistency becomes more expensive to violate.

In [ ]:
totals_a2 = np.vstack([adv_a2 + lam_a2 * cyc_a2 for lam_a2 in lambdas_a2])  # Build all total-loss rows for plotting.
print("totals at λ=1:", np.round(totals_a2[2], 3))  # Inspect one row numerically.

▶ What you'll see: each λ creates a different ranking of the same candidate translators.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a line plot for each candidate.
for j_a2 in range(3):  # Plot model losses across λ.
    plt.plot(lambdas_a2, totals_a2[:, j_a2], marker="o", label=f"model {j_a2}")  # Draw one curve.
plt.title("Advanced 2: λ_cycle tradeoff")  # Title the sweep.
plt.xlabel("λ_cycle")  # Label the cycle weight.
plt.ylabel("total loss")  # Label the objective.
plt.legend()  # Show model labels.
plt.show()  # Display the plot.

▶ What you'll see: increasing λ changes which model has the lowest total loss.

👀 Takeaway: too little cycle weight allows content drift; too much can over-constrain translation.

### Advanced 3 — Inspect style truncation toward an average style

**Goal.** Pull style vectors toward the average style, because StyleGAN's truncation trick trades diversity for more typical samples. We build it in 4 steps.

In [ ]:
styles_a3 = np.array([[2.0, 1.5], [1.0, 0.5], [-1.5, 2.0], [0.5, -1.0]])  # Define four style vectors.
mean_style_a3 = styles_a3.mean(axis=0)  # Compute the average style.
psi_a3 = 0.5  # Choose truncation strength.
print("mean style:", np.round(mean_style_a3, 3))  # Inspect the center of styles.

▶ What you'll see: the average style is the center that truncation pulls toward.

In [ ]:
truncated_a3 = mean_style_a3 + psi_a3 * (styles_a3 - mean_style_a3)  # Move each style halfway toward the mean.
dist_before_a3 = np.linalg.norm(styles_a3 - mean_style_a3, axis=1).mean()  # Average distance before truncation.
dist_after_a3 = np.linalg.norm(truncated_a3 - mean_style_a3, axis=1).mean()  # Average distance after truncation.
print("mean distance before/after:", round(float(dist_before_a3), 3), round(float(dist_after_a3), 3))  # Inspect diversity shrinkage.
assert round(float(dist_after_a3 / dist_before_a3), 3) == 0.5  # Verify ψ halves distance to the mean.

▶ What you'll see: truncation halves the average distance from the style center.

In [ ]:
x_a3 = np.linspace(-1, 1, 80)  # Define a base activation coordinate.
signal_before_a3 = style_modulate(x_a3, styles_a3[0, 0], styles_a3[0, 1])  # Generate with an original style.
signal_after_a3 = style_modulate(x_a3, truncated_a3[0, 0], truncated_a3[0, 1])  # Generate with a truncated style.
print("style 0 before/after:", np.round(styles_a3[0], 3), np.round(truncated_a3[0], 3))  # Inspect one changed style.

▶ What you'll see: the first style's scale and bias move closer to the population mean.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a signal comparison plot.
plt.plot(signal_before_a3, label="original style")  # Plot original styled signal.
plt.plot(signal_after_a3, label="truncated style")  # Plot truncated styled signal.
plt.title("Advanced 3: truncation reduces extremeness")  # Title the plot.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the truncated signal is less extreme because its style vector moved toward average.

👀 Takeaway: truncation improves typicality by shrinking style variation, but that same shrinkage reduces diversity.

### Advanced 4 — Compose AdaIN with layer mixing

**Goal.** Apply different style statistics to coarse and fine feature channels, because StyleGAN-like generators can control appearance at multiple scales. We build it in 4 steps.

In [ ]:
coarse_map_a4 = np.tile(np.linspace(0, 1, 8), (8, 1))  # Define a smooth coarse feature map.
fine_map_a4 = ((np.indices((8, 8)).sum(axis=0) % 2) * 2 - 1).astype(float)  # Define a checkerboard fine feature map.
content_a4 = np.stack([coarse_map_a4, fine_map_a4])  # Stack as two channels.
print("content shape:", content_a4.shape)  # Inspect channel and spatial dimensions.

▶ What you'll see: two channels: one smooth ramp and one high-frequency checkerboard.

In [ ]:
style_a_mean_a4 = np.array([0.0, 0.0])  # Style A means.
style_a_std_a4 = np.array([1.0, 0.2])  # Style A: weak fine texture.
style_b_mean_a4 = np.array([0.5, 0.0])  # Style B means.
style_b_std_a4 = np.array([0.4, 2.0])  # Style B: strong fine texture.
print("fine std A/B:", style_a_std_a4[1], style_b_std_a4[1])  # Inspect fine texture contrast.

▶ What you'll see: style B asks for ten times more fine-channel contrast than style A.

In [ ]:
mixed_mean_a4 = np.array([style_a_mean_a4[0], style_b_mean_a4[1]])  # Take coarse mean from A and fine mean from B.
mixed_std_a4 = np.array([style_a_std_a4[0], style_b_std_a4[1]])  # Take coarse std from A and fine std from B.
styled_a4 = adain(content_a4, mixed_mean_a4, mixed_std_a4)  # Apply mixed AdaIN statistics.
print("styled stds:", np.round(styled_a4.std(axis=(1, 2)), 3))  # Inspect achieved channel contrasts.
assert np.allclose(np.round(styled_a4.std(axis=(1, 2)), 3), [1.0, 2.0])  # Verify mixed stds.

▶ What you'll see: the coarse channel uses A's contrast while the fine channel uses B's contrast.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.7))  # Create heatmaps for the two channels.
ax[0].imshow(styled_a4[0], cmap="viridis"); ax[0].set_title("coarse styled")  # Show coarse channel.
ax[1].imshow(styled_a4[1], cmap="coolwarm"); ax[1].set_title("fine styled")  # Show fine channel.
plt.suptitle("Advanced 4: AdaIN with mixed styles")  # Title the figure.
plt.show()  # Display the figure.

▶ What you'll see: the coarse ramp and fine checkerboard have independently chosen style strengths.

👀 Takeaway: AdaIN and layer mixing combine naturally: each channel or layer can receive its own style statistics.

### Advanced 5 — Train a tiny cycle-consistent linear translator

**Goal.** Optimize a small return map by gradient descent, because CycleGAN training adjusts translators to reduce cycle-consistency error. We build it in 5 steps.

In [ ]:
x_a5 = np.array([[-1.0], [0.0], [1.0], [2.0]])  # Define 1-D source samples as column vectors.
y_a5 = 2.0 * x_a5 + 1.0  # Define a fixed forward translator G(x)=2x+1.
w_a5 = 0.2  # Initialize F(y)=w*y+b with a poor slope.
b_a5 = 0.0  # Initialize a poor bias.
print("initial w,b:", w_a5, b_a5)  # Inspect return-map parameters.

▶ What you'll see: the return map starts far from the inverse F(y)=0.5y-0.5.

In [ ]:
lr_a5 = 0.05  # Choose a small learning rate.
losses_a5 = []  # Store cycle losses during optimization.
for step_a5 in range(120):  # Run gradient descent on mean squared cycle error for smooth gradients.
    x_back_a5 = w_a5 * y_a5 + b_a5  # Compute F(G(x)).
    err_a5 = x_back_a5 - x_a5  # Compute return-trip error.
    losses_a5.append(float(np.mean(err_a5 ** 2)))  # Store MSE cycle loss.
    grad_w_a5 = float(2 * np.mean(err_a5 * y_a5))  # Derivative of mean squared error with respect to w.
    grad_b_a5 = float(2 * np.mean(err_a5))  # Derivative with respect to b.
    w_a5 -= lr_a5 * grad_w_a5  # Descend on w.
    b_a5 -= lr_a5 * grad_b_a5  # Descend on b.
print("trained w,b:", round(w_a5, 3), round(b_a5, 3))  # Inspect learned inverse parameters.

▶ What you'll see: the learned return map moves toward slope 0.5 and bias -0.5.

In [ ]:
x_back_final_a5 = w_a5 * y_a5 + b_a5  # Compute final returned samples.
final_l1_a5 = cycle_l1(x_a5, x_back_final_a5)  # Measure L1 cycle consistency after training.
print("final cycle L1:", round(final_l1_a5, 3))  # Inspect return-trip quality.
assert final_l1_a5 < 0.06  # Verify optimization made the cycle nearly consistent.

▶ What you'll see: the final average absolute return-trip error is small.

In [ ]:
print("loss start/end:", round(losses_a5[0], 3), round(losses_a5[-1], 5))  # Inspect convergence numerically.
assert losses_a5[-1] < losses_a5[0]  # Verify the cycle objective decreased.

▶ What you'll see: the cycle loss falls substantially from its initial value.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a convergence plot.
plt.plot(losses_a5, color="teal")  # Plot cycle MSE over steps.
plt.title("Advanced 5: learning a return translator")  # Title the learning curve.
plt.xlabel("gradient step")  # Label optimization step.
plt.ylabel("cycle MSE")  # Label loss scale.
plt.show()  # Display the curve.

▶ What you'll see: the curve slopes downward as the return translator learns the inverse.

👀 Takeaway: cycle consistency is not just a diagnostic; it supplies gradients that train the translator to preserve recoverable content.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

StyleGAN controls how features are drawn; CycleGAN learns translations when paired examples are unavailable.

Style modulation steers features continuously, while cycle consistency constrains unpaired translation with a return trip. The key is control without paired supervision.

Save a copy to Drive to edit. This notebook is CPU-only, seeded, and designed to be inspected before you run any cell.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

SEED = 1007
rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-a))


def stable_log(x):
    return np.log(np.clip(x, 1e-8, 1.0))


def standardize(X):
    X = np.asarray(X, dtype=float)
    center = X.mean(axis=0, keepdims=True)
    scale = X.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    return (X - center) / scale


def pca_project_reconstruct(X, latent_dim, shrink=1.0):
    X = np.asarray(X, dtype=float)
    latent_dim = int(min(latent_dim, X.shape[0] - 1, X.shape[1]))
    latent_dim = max(latent_dim, 1)
    model = PCA(n_components=latent_dim, random_state=SEED)
    Z = model.fit_transform(X)
    Z = Z * shrink
    X_hat = model.inverse_transform(Z)
    return model, Z, X_hat


def reconstruction_error(X, X_hat):
    return float(0.5 * np.mean(np.sum((X - X_hat) ** 2, axis=1)))


def rbf_two_sample_distance(A, B, gamma=None):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    both = np.vstack([A, B])
    if gamma is None:
        d = pairwise_distances(both[: min(len(both), 80)])
        positive = d[d > 0]
        width = np.median(positive) if positive.size else 1.0
        gamma = 1.0 / (2.0 * width ** 2 + 1e-8)
    Kaa = np.exp(-gamma * pairwise_distances(A, A) ** 2).mean()
    Kbb = np.exp(-gamma * pairwise_distances(B, B) ** 2).mean()
    Kab = np.exp(-gamma * pairwise_distances(A, B) ** 2).mean()
    return float(Kaa + Kbb - 2.0 * Kab)


def sample_latent_decode(model, Z, n_samples, noise_scale=0.75):
    mu = Z.mean(axis=0)
    sigma = Z.std(axis=0) + 1e-6
    Z_new = rng.normal(mu, noise_scale * sigma, size=(n_samples, Z.shape[1]))
    return model.inverse_transform(Z_new)


def make_digit_hard_set(limit=240):
    digits = load_digits()
    X = digits.data.astype(float) / 16.0
    y = digits.target
    X = X[:limit]
    y = y[:limit]
    rows = []
    for i, row in enumerate(X):
        image = row.reshape(8, 8)
        shift = (i % 3) - 1
        moved = np.roll(image, shift=shift, axis=1)
        nuisance = 0.20 * np.sin(np.linspace(0.0, np.pi, 64) + 0.3 * y[i])
        noisy = np.clip(moved.reshape(-1) + nuisance + rng.normal(0.0, 0.06, 64), 0.0, 1.0)
        rows.append(noisy)
    hard = np.asarray(rows)
    interactions = hard[:, :16] * hard[:, 16:32]
    return np.hstack([hard, interactions])


def build_f9_ladder():
    X1 = rng.normal(0.0, 1.0, size=(160, 1))
    X2, y2 = make_moons(n_samples=180, noise=0.06, random_state=SEED)
    means = np.array([[-2.0, 0.0], [1.5, 1.3], [1.2, -1.4]])
    parts = []
    labels = []
    for k, mean in enumerate(means):
        cov = np.array([[0.10 + 0.05 * k, 0.03], [0.03, 0.18]])
        draw = rng.multivariate_normal(mean, cov, size=70)
        parts.append(draw)
        labels.extend([k] * len(draw))
    X3 = np.vstack(parts)
    y3 = np.asarray(labels)
    digits = load_digits()
    X4 = digits.data.astype(float) / 16.0
    y4 = digits.target
    X4 = X4[:240]
    y4 = y4[:240]
    X5 = make_digit_hard_set(limit=240)
    y5 = y4.copy()
    return [
        {"rung": "D1", "name": "1-D Gaussian", "X": standardize(X1), "y": None, "image_shape": None},
        {"rung": "D2", "name": "2-D two moons", "X": standardize(X2), "y": y2, "image_shape": None},
        {"rung": "D3", "name": "3-component mixture", "X": standardize(X3), "y": y3, "image_shape": None},
        {"rung": "D4", "name": "sklearn digits", "X": standardize(X4), "y": y4, "image_shape": (8, 8)},
        {"rung": "D5", "name": "noisy shifted digits with interactions", "X": standardize(X5), "y": y5, "image_shape": (8, 8)},
    ]


def preview_ladder(ladder):
    for item in ladder:
        X = item["X"]
        y = item["y"]
        classes = "none" if y is None else len(np.unique(y))
        print(f"{item['rung']} | {item['name']} | shape={X.shape} | classes={classes}")
        print(np.round(X[:2, : min(6, X.shape[1])], 3))


def first_two_dimensions(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.column_stack([X[:, 0], np.zeros(len(X))])
    if X.shape[1] == 2:
        return X
    model = PCA(n_components=2, random_state=SEED)
    return model.fit_transform(X)


def panel_axis(ax, X, title, image_shape=None):
    X = np.asarray(X, dtype=float)
    if image_shape is not None and X.shape[1] >= image_shape[0] * image_shape[1]:
        side = image_shape[0] * image_shape[1]
        mosaic = X[:16, :side].reshape(16, image_shape[0], image_shape[1])
        rows = []
        for start in range(0, 16, 4):
            rows.append(np.hstack(mosaic[start:start + 4]))
        ax.imshow(np.vstack(rows), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        xy = first_two_dimensions(X[:120])
        ax.scatter(xy[:, 0], xy[:, 1], s=10, alpha=0.75)
    ax.set_title(title, fontsize=9)


def plot_generated_panels(results, metric_name):
    fig, axes = plt.subplots(2, len(results), figsize=(3.0 * len(results), 5.0))
    for j, result in enumerate(results):
        panel_axis(axes[0, j], result["generated"], result["rung"], result.get("image_shape"))
        axes[1, j].bar([0], [result["metric"]])
        axes[1, j].set_xticks([0])
        axes[1, j].set_xticklabels([metric_name], rotation=30, ha="right")
        axes[1, j].set_title(f"{result['metric']:.3f}", fontsize=9)
    fig.tight_layout()
    plt.show()
    plt.figure(figsize=(6, 3))
    plt.plot([r["rung"] for r in results], [r["metric"] for r in results], marker="o")
    plt.ylabel(metric_name)
    plt.xlabel("dataset complexity")
    plt.title(f"{metric_name} vs complexity")
    plt.grid(True, alpha=0.3)
    plt.show()


## The concept, built once: style modulation and cycle consistency

The lesson formulas are $y=s\cdot x+b$ and $L_{cyc}=\lVert F(G(x))-x\rVert_1$. We assert the exact activation $1.5\cdot2+0.4=3.400$ and a direct cycle calculation.

In [ ]:
def style_mod_and_cycle_loss(x, scale, bias, returned):
    styled = float(scale * x + bias)
    cycle = float(abs(returned - x))
    return styled, cycle

styled, cycle = style_mod_and_cycle_loss(2.0, 1.5, 0.4, 2.0)
illustrative_cycle = abs(2.2 - 2.0)
print(round(styled, 3), round(cycle, 4), round(illustrative_cycle, 1))
assert round(styled, 3) == 3.400
assert round(cycle, 4) == 0.0000
assert round(illustrative_cycle, 1) == 0.2

The reusable method applies a style affine map in a latent PCA space, applies an approximate inverse map, and reports cycle-consistency reconstruction error across rungs.

In [ ]:
def run_topic_model(item):
    X = item["X"]
    latent_dim = 1 if X.shape[1] <= 2 else min(10, X.shape[1] // 4)
    model, Z, X_hat = pca_project_reconstruct(X, latent_dim=latent_dim)
    scales = rng.normal(1.05, 0.08, size=Z.shape[1])
    biases = rng.normal(0.0, 0.15, size=Z.shape[1])
    styled_Z = Z * scales + biases
    returned_Z = (styled_Z - biases) / scales
    generated = model.inverse_transform(styled_Z[:40])
    cycled = model.inverse_transform(returned_Z)
    metric = reconstruction_error(X, cycled)
    return {
        "rung": item["rung"],
        "name": item["name"],
        "metric": metric,
        "generated": generated,
        "image_shape": item["image_shape"],
    }

## The D1-D5 generative ladder

Family F9 uses an inline ladder rather than a shared helper: D1 is a 1-D Gaussian, D2 is two moons, D3 is a mixture, D4 is bundled `sklearn.datasets.load_digits`, and D5 is a harder no-download real/synthetic digits set with shifts, nuisance variation, and feature interactions.

In [ ]:
ladder = build_f9_ladder()
preview_ladder(ladder)

In [ ]:
results = []
for item in ladder:
    result = run_topic_model(item)
    results.append(result)
    print(f"{result['rung']} {result['name']}: cycle_error={result['metric']:.4f}")

In [ ]:
plot_generated_panels(results, "cycle error")

## Pitfall on D5: dropping cycle loss causes content drift

An adversarial-only map can move samples toward a target style while losing content. Adding the inverse cycle penalizes that drift.

In [ ]:
d5 = ladder[-1]
X = d5["X"]
model, Z, X_hat = pca_project_reconstruct(X, latent_dim=10)
drift = rng.normal(0.0, 0.9, size=Z.shape)
no_cycle_Z = Z * 1.2 + drift
cycle_Z = (no_cycle_Z - drift) / 1.2
no_cycle_back = model.inverse_transform(no_cycle_Z)
cycle_back = model.inverse_transform(cycle_Z)
no_cycle_error = reconstruction_error(X, no_cycle_back)
cycle_error = reconstruction_error(X, cycle_back)
print(f"without cycle loss content drift={no_cycle_error:.4f}")
print(f"with cycle consistency drift={cycle_error:.4f}")

## Evaluate it + practice

- Metric: track reconstruction/cycle-consistency error on every rung and compare against a no-skill baseline that samples noisy training examples.
- Sanity check: generated samples should match the rough support of D1-D3 before you trust D4-D5 panels.
- Ablation: drop the inverse cycle and measure content drift after style translation.
- Failure signal: D5 improves visually while the summary metric worsens, or one mode repeats across many generated panels.
- Reproducibility: keep the seed fixed, then change only one knob at a time.

Practice 1: Change the latent dimension or codebook size and predict which rung changes most.

Practice 2: Replace the D5 nuisance strength with a smaller value and compare the metric curve.

Practice 3: Add one diagnostic printout that would catch the named pitfall before plotting.